# Data Loading and Initial Inspection

This notebook loads the raw retail dataset and performs a structured inspection before any cleaning or feature engineering begins.

## Objectives
- Load the source CSV safely with encoding fallback
- Inspect structure, schema, missing values, and duplicates
- Identify suspicious columns and data quality issues
- Document what must be addressed in the cleaning stage


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from scripts.data_utils import read_csv_with_fallback

raw_path = project_root / "data" / "raw" / "superstore_raw.csv"
df, used_encoding = read_csv_with_fallback(raw_path)

print(f"Loaded dataset with encoding: {used_encoding}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")


## Dataset Preview


In [ ]:
df.head()


## Column Names and Data Types


In [ ]:
pd.DataFrame(
    {
        "column_name": df.columns,
        "dtype": df.dtypes.astype(str).values,
    }
)


In [ ]:
df.info()


## Summary Statistics


In [ ]:
df.describe(include="all").transpose()


## Missing Values and Duplicates

These checks matter because missing fields and duplicated rows can distort revenue totals, customer frequency, churn labels, and forecast inputs.


In [ ]:
missing_summary = (
    pd.DataFrame(
        {
            "missing_count": df.isna().sum(),
            "missing_percent": (df.isna().mean() * 100).round(2),
        }
    )
    .sort_values("missing_count", ascending=False)
)
missing_summary[missing_summary["missing_count"] > 0]


In [ ]:
duplicate_rows = df.duplicated().sum()
suspicious_columns = [
    col for col in df.columns
    if "unnamed" in str(col).lower() or str(col).strip() == ""
]

print(f"Duplicate rows: {duplicate_rows:,}")
print("Suspicious columns:", suspicious_columns if suspicious_columns else "None found")


## Initial Observations

The raw file includes naming inconsistencies, likely utility columns that do not belong in the final analytics model, and date fields that still need conversion. The next notebook standardizes the schema and prepares a reusable cleaned dataset.
